Import the libraries


In [16]:
# ---------- ALL IMPORTS (one place) ----------
import random
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait



In [17]:
'''1. Settings'''
KEYWORD = "Data Scientist"
LOCATION = "India"
TARGET = 100

# find the project root (the folder that contains DATA)
ROOT = Path.cwd()
if not (ROOT / "DATA").exists():
    ROOT = ROOT.parent
OUT = ROOT / "DATA" / "jobs_cards.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

url = "https://www.linkedin.com/jobs/search?" + urlencode(
    {"keywords": KEYWORD, "location": LOCATION}
)
print("Project root:", ROOT)
print("Will save to:", OUT)

Project root: C:\Linkedin_Job_Analysis
Will save to: C:\Linkedin_Job_Analysis\DATA\jobs_cards.csv


## Scraping log
- Location for all searches: India
- Keywords scraped (one at a time, by changing KEYWORD in Cell 1 and running Cells 4 and 5):
  1. Data Analyst
  2. Data Scientist
  3. Data Engineer
  4. Business Analyst
  5. Power BI Developer
- Detail scraping (Cell 8) completed for all 332 jobs.
- Known issue: seniority/employment type/industries and some titles were missing; fixed by the repair cell (Cell 12).
- Re-run only the cells for a NEW keyword. Do not re-run Cells 4/5/8 for the old ones.

In [18]:
#2 functions (get_text, parse_card, load_more_cards)
def load_more_cards(driver, target):
    stuck = 0
    last_count = 0
    while True:
        count = len(driver.find_elements(By.CSS_SELECTOR, "div.base-card"))
        print("Loaded:", count)
        if count >= target:
            break
        if count == last_count:
            stuck += 1
            if stuck >= 4:
                print("No more jobs are loading. Stopping.")
                break
        else:
            stuck = 0
        last_count = count

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(1.5, 2.5))
        try:
            button = driver.find_element(By.CSS_SELECTOR, "button.infinite-scroller__show-more-button")
            if button.is_displayed():
                driver.execute_script("arguments[0].click();", button)
                time.sleep(random.uniform(1.5, 2.5))
        except NoSuchElementException:
            pass
    return driver.find_elements(By.CSS_SELECTOR, "div.base-card")[:target]


def get_text(card, selector):
    try:
        return card.find_element(By.CSS_SELECTOR, selector).text.strip()
    except NoSuchElementException:
        return ""


def parse_card(card):
    """Turn one job card into a dictionary (one future CSV row)."""
    job = {
        "title": get_text(card, ".base-search-card__title"),
        "company": get_text(card, ".base-search-card__subtitle"),
        "location": get_text(card, ".job-search-card__location"),
        "posted_date": "",
        "posted_text": "",
        "job_url": "",
        "job_id": "",
    }
    try:
        t = card.find_element(By.CSS_SELECTOR, "time")
        job["posted_date"] = t.get_attribute("datetime") or ""
        job["posted_text"] = t.text.strip()
    except NoSuchElementException:
        pass
    try:
        href = card.find_element(By.CSS_SELECTOR, "a.base-card__full-link").get_attribute("href")
        job["job_url"] = href.split("?")[0]
    except NoSuchElementException:
        pass

    urn = card.get_attribute("data-entity-urn") or ""
    match = re.search(r"(\d{6,})", urn) or re.search(r"(\d{6,})$", job["job_url"])
    job["job_id"] = match.group(1) if match else ""
    return job

In [19]:
#3 Run scraper for cards
driver = webdriver.Chrome()
driver.set_window_size(1400, 1000)

try:
    driver.get(url)
    time.sleep(4)
    cards = load_more_cards(driver, TARGET)
    jobs = [parse_card(c) for c in cards]      # read the data BEFORE closing the browser
finally:
    driver.quit()

print("Jobs scraped:", len(jobs))

Loaded: 60
Loaded: 70
Loaded: 80
Loaded: 90
Loaded: 100
Jobs scraped: 100


In [20]:
#4 Save cards
new_cards = pd.DataFrame(jobs)
new_cards["search_keyword"] = KEYWORD
new_cards["search_location"] = LOCATION
new_cards = new_cards[new_cards["job_id"] != ""]          # drop cards without an id

print("New cards scraped:", len(new_cards))
print("\nEmpty values per column:")
print((new_cards == "").sum())

if OUT.exists():
    old = pd.read_csv(OUT, dtype={"job_id": str})
    combined = pd.concat([old, new_cards], ignore_index=True)
else:
    combined = new_cards

combined = combined.drop_duplicates(subset="job_id", keep="first")
combined.to_csv(OUT, index=False)
print("\nTotal unique cards in file:", len(combined))
combined.head()

New cards scraped: 100

Empty values per column:
title              0
company            0
location           0
posted_date        0
posted_text        0
job_url            0
job_id             0
search_keyword     0
search_location    0
dtype: int64

Total unique cards in file: 355


,title,company,location,posted_date,posted_text,job_url,job_id,search_keyword,search_location
0,Senior Data Analyst,EXL,"Bengaluru, Karnataka, India",2026-10-02,1 day ago,https://in.linkedin.com/jobs/view/senior-data-...,4446724293,Data Analyst,India
1,Senior Data Analyst,Finkraft,"Bangalore Urban, Karnataka, India",2026-06-27,3 months ago,https://in.linkedin.com/jobs/view/senior-data-...,4433474003,Data Analyst,India
2,Senior Data Analyst,Guidehouse,"Chennai, Tamil Nadu, India",2026-09-24,1 week ago,https://in.linkedin.com/jobs/view/senior-data-...,4471438516,Data Analyst,India
3,Senior Data Analyst I with Python Experience,ICE,"Pune Division, Maharashtra, India",2026-09-16,2 weeks ago,https://in.linkedin.com/jobs/view/senior-data-...,4459008614,Data Analyst,India
4,Assistant Manager,EXL,"Bengaluru, Karnataka, India",2026-10-02,1 day ago,https://in.linkedin.com/jobs/view/assistant-ma...,4446587231,Data Analyst,India


In [21]:
#5 "Detail functions"
RAW_OUT = ROOT / "DATA" / "jobs_raw.csv"

COLUMNS = ["job_id", "title", "company", "location", "posted_date", "posted_text",
           "job_url", "seniority_level", "employment_type", "job_function",
           "industries", "description", "search_keyword", "search_location", "scraped_at"]

CRITERIA_MAP = {"seniority level": "seniority_level", "employment type": "employment_type",
                "job function": "job_function", "industries": "industries"}

cards_df = pd.read_csv(OUT, dtype={"job_id": str}).fillna("")
print("Cards loaded:", len(cards_df))


def blocked(driver):
    u = driver.current_url.lower()
    return "authwall" in u or "/login" in u or "/checkpoint" in u


def read_job_details(driver, url):
    """Open one job page and return its details. Returns None if LinkedIn blocks us."""
    details = {"description": "", "seniority_level": "", "employment_type": "",
               "job_function": "", "industries": ""}
    driver.get(url)
    time.sleep(random.uniform(2, 4))
    if blocked(driver):
        return None
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.show-more-less-html__markup")))
    except TimeoutException:
        return details

    desc = driver.find_element(By.CSS_SELECTOR, "div.show-more-less-html__markup")
    raw_text = desc.get_attribute("textContent") or ""
    clean_text = re.sub(r"\s+", " ", raw_text)
    details["description"] = clean_text.strip()

    # wait for the criteria section too, instead of reading it immediately
    try:
        WebDriverWait(driver, 8).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "li.description__job-criteria-item")))
    except TimeoutException:
        pass

    for item in driver.find_elements(By.CSS_SELECTOR, "li.description__job-criteria-item"):
        header = get_text(item, "h3.description__job-criteria-subheader").lower()
        value = get_text(item, "span.description__job-criteria-text")
        if header in CRITERIA_MAP:
            details[CRITERIA_MAP[header]] = value
    return details


def parse_page_title(page_title):
    """'Data Analyst at EXL — Pune, India | LinkedIn Jobs' -> (title, company, location)"""
    t = page_title.replace(" | LinkedIn Jobs", "").replace(" | LinkedIn", "").strip()
    location = company = ""
    if " — " in t:
        t, location = t.rsplit(" — ", 1)
    if " at " in t:
        t, company = t.rsplit(" at ", 1)
    return t.strip(), company.strip(), location.strip()


def read_header(driver):
    """Title, company and location from the job page that is currently open."""
    page_title, company, location = parse_page_title(driver.title)
    title = ""
    for sel in ["h1.top-card-layout__title", "h2.top-card-layout__title"]:
        title = get_text(driver, sel)
        if title:
            break
    return {"title": title or page_title, "company": company, "location": location}

Cards loaded: 355


In [53]:
#6Test on 3 jobs
driver = webdriver.Chrome()
driver.set_window_size(1400, 1000)
try:
    for _, row in cards_df.head(3).iterrows():
        d = read_job_details(driver, row["job_url"])
        if d is None:
            print("Blocked by login wall"); break
        print(row["title"], "|", row["company"])
        print("   seniority:", d["seniority_level"], "| type:", d["employment_type"])
        print("   function:", d["job_function"], "| industries:", d["industries"])
        print("   description length:", len(d["description"]), "characters\n")
finally:
    driver.quit()

Senior Data Analyst | EXL
   seniority: Mid-Senior level | type: Full-time
   function:  | industries: Business Consulting and Services
   description length: 502 characters

Senior Data Analyst | Finkraft
   seniority: Mid-Senior level | type: Full-time
   function:  | industries: Financial Services
   description length: 2118 characters

Senior Data Analyst | Guidehouse
   seniority: Associate | type: Full-time
   function:  | industries: Business Consulting and Services
   description length: 4619 characters



In [42]:
#7 Scrape all job details
already_done = set()
if RAW_OUT.exists():
    already_done = set(pd.read_csv(RAW_OUT, dtype={"job_id": str})["job_id"])
print("Already saved:", len(already_done))

todo = cards_df[~cards_df["job_id"].isin(already_done)]
print("To scrape now:", len(todo))

driver = webdriver.Chrome()
driver.set_window_size(1400, 1000)
saved = 0
try:
    for _, row in todo.iterrows():
        details = read_job_details(driver, row["job_url"])
        if details is None:
            print("Login wall appeared. Stopping safely; your data so far is saved.")
            break
        job = row.to_dict()
        job.update(details)
        job["scraped_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        pd.DataFrame([job])[COLUMNS].to_csv(
            RAW_OUT, mode="a", index=False, header=not RAW_OUT.exists())
        saved += 1
        print(f"[{saved}/{len(todo)}] {job['title']} | {job['company']}")
finally:
    driver.quit()
    print("Done. Saved this run:", saved)

Already saved: 126
To scrape now: 206
[1/206]  | 
[2/206]  | 
[3/206]  | 
[4/206]  | 
[5/206]  | 
[6/206]  | 
[7/206]  | 
[8/206]  | 
[9/206]  | 
[10/206]  | 
[11/206]  | 
[12/206]  | 
[13/206]  | 
[14/206]  | 
[15/206]  | 
[16/206]  | 
[17/206]  | 
[18/206]  | 
[19/206]  | 
[20/206]  | 
[21/206]  | 
[22/206]  | 
[23/206]  | 
[24/206]  | 
[25/206]  | 
[26/206]  | 
[27/206]  | 
[28/206]  | 
[29/206]  | 
[30/206]  | 
[31/206]  | 
[32/206]  | 
[33/206]  | 
[34/206]  | 
[35/206]  | 
[36/206]  | 
[37/206]  | 
[38/206]  | 
[39/206]  | 
[40/206]  | 
[41/206]  | 
[42/206]  | 
[43/206]  | 
[44/206]  | 
[45/206]  | 
[46/206]  | 
[47/206]  | 
[48/206]  | 
[49/206]  | 
[50/206]  | 
[51/206]  | 
[52/206]  | 
[53/206]  | 
[54/206]  | 
[55/206]  | 
[56/206]  | 
[57/206] FBS Data Engineer | Capgemini
[58/206] Senior Data Engineer | Cloudwick
[59/206] Data Engineer | NK Securities Research
[60/206] Data Engineer Consultant II | Allstate India
[61/206] Sr. Business Analyst | VGreen Technology Solutions 

In [22]:
#8 Check raw data
raw = pd.read_csv(RAW_OUT, dtype={"job_id": str})
print("Shape:", raw.shape)
print("\nEmpty values per column:")
print(raw.isna().sum())
print("\nDescription length (characters):")
print(raw["description"].fillna("").str.len().describe().round(0))
raw[["title", "company", "seniority_level", "employment_type", "description"]].head()

Shape: (332, 15)

Empty values per column:
job_id               0
title                0
company              0
location             0
posted_date          0
posted_text         56
job_url              0
seniority_level      0
employment_type      0
job_function       332
industries           0
description          0
search_keyword       0
search_location      0
scraped_at           0
dtype: int64

Description length (characters):
count      332.0
mean      3576.0
std       2098.0
min        373.0
25%       2029.0
50%       3496.0
75%       4661.0
max      15520.0
Name: description, dtype: float64


,title,company,seniority_level,employment_type,description
0,Senior Data Analyst,EXL,Mid-Senior level,Full-time,Title : Senior Data AnalystLocation : Bangalor...
1,Senior Data Analyst,Finkraft,Mid-Senior level,Full-time,About Finkraft.aiFinkraft.ai is a fast-growing...
2,Senior Data Analyst,Guidehouse,Associate,Full-time,Job FamilyEBO Management Information System (I...
3,Senior Data Analyst I with Python Experience,ICE,Not Applicable,Full-time,Job PurposeICE is the leading cloud-based plat...
4,Assistant Manager,EXL,Entry level,Full-time,Title - Data AnalystLocation - Bangalore (WFO)...


In [32]:
#9 "Check 1 / Check 2"
cards_now = pd.read_csv(OUT, dtype={"job_id": str})
saved_now = pd.read_csv(RAW_OUT, dtype={"job_id": str})
print("Cards in jobs_cards.csv:", len(cards_now))
print("Jobs in jobs_raw.csv:   ", len(saved_now))
print("Still to scrape:        ", (~cards_now["job_id"].isin(saved_now["job_id"])).sum())

# Check 2: what does LinkedIn really call the job function field?
driver = webdriver.Chrome()
try:
    driver.get(cards_now.loc[0, "job_url"])
    time.sleep(4)
    items = driver.find_elements(By.CSS_SELECTOR, "li.description__job-criteria-item")
    print("\nCriteria items found:", len(items))
    for item in items:
        print(repr(item.get_attribute("textContent").strip()))
finally:
    driver.quit()

Cards in jobs_cards.csv: 60
Jobs in jobs_raw.csv:    60
Still to scrape:         0

Criteria items found: 3
'Seniority level\n          \n          \n            Mid-Senior level'
'Employment type\n          \n          \n              Full-time'
'Industries\n            \n            \n            Business Consulting and Services'


In [44]:
#10. "diagnostic cell"
raw = pd.read_csv(RAW_OUT, dtype={"job_id": str})
raw["no_title"] = raw["title"].isna()
raw["no_criteria"] = raw["seniority_level"].isna()

print("Do the two problems overlap?")
print(pd.crosstab(raw["no_title"], raw["no_criteria"]), "\n")

print("Problems by search keyword:")
print(raw.groupby("search_keyword")[["no_title", "no_criteria"]].sum())
print(raw["search_keyword"].value_counts(), "\n")

# Open ONE job that is missing seniority and inspect the page
bad_url = raw.loc[raw["no_criteria"], "job_url"].iloc[0]
print("Inspecting:", bad_url)

driver = webdriver.Chrome()
driver.set_window_size(1400, 1000)
try:
    driver.get(bad_url)
    time.sleep(6)
    print("Page title:", driver.title)
    print("Criteria items found:", len(driver.find_elements(By.CSS_SELECTOR, "li.description__job-criteria-item")))
    print("'Seniority level' appears in page:", "Seniority level" in driver.page_source)
    print("Class names containing 'criteria':")
    print(driver.execute_script(
        "return [...new Set([...document.querySelectorAll('[class*=criteria]')].map(e => e.className))].slice(0, 10)"))
finally:
    driver.quit()

Do the two problems overlap?
no_criteria  False  True 
no_title                 
False           60    216
True             0     56 

Problems by search keyword:
                    no_title  no_criteria
search_keyword                           
Business Analyst           0           86
Data Analyst               0            0
Data Engineer             56           60
Data Scientist             0           66
Power BI Developer         0           60
search_keyword
Business Analyst      86
Data Scientist        66
Data Analyst          60
Data Engineer         60
Power BI Developer    60
Name: count, dtype: int64 

Inspecting: https://in.linkedin.com/jobs/view/data-scientist-data-science-data-scientist-at-exl-4446725483
Page title: Data Scientist-Data Science-Data Scientist at EXL — Gurugram, Haryana, India | LinkedIn Jobs
Criteria items found: 3
'Seniority level' appears in page: True
Class names containing 'criteria':
['description__job-criteria-list', 'description__job-criteria-it

In [56]:
import shutil

LIMIT = 3            # 3 = dry run (nothing saved).  Set to None for the real repair.
SAVE = LIMIT is None

if SAVE:
    shutil.copy(RAW_OUT, ROOT / "DATA" / "jobs_raw_backup.csv")      # safety copy

raw = pd.read_csv(RAW_OUT, dtype={"job_id": str})
needs = (raw["title"].isna() | raw["company"].isna() | raw["location"].isna()
         | raw["seniority_level"].isna())
todo_idx = list(raw.index[needs])
print("Rows needing repair:", len(todo_idx))
if LIMIT:
    todo_idx = todo_idx[:LIMIT]

driver = webdriver.Chrome()
driver.set_window_size(1400, 1000)
try:
    for n, i in enumerate(todo_idx, start=1):
        details = read_job_details(driver, raw.at[i, "job_url"])
        if details is None:
            print("Login wall appeared. Stopping safely.")
            break

        # DEBUG: what did the scraper actually read from this page?
        print("   DEBUG details ->",
              "seniority:", repr(details["seniority_level"]),
              "| type:", repr(details["employment_type"]),
              "| industries:", repr(details["industries"]),
              "| description length:", len(details["description"]))

        info = read_header(driver)
        for col in ["seniority_level", "employment_type", "industries"]:
            info[col] = details[col]

        for col, val in info.items():
            if val and pd.isna(raw.at[i, col]):          # only fill blanks
                raw.at[i, col] = val

        print(f"[{n}/{len(todo_idx)}] {raw.at[i, 'title']} | {raw.at[i, 'company']} | "
              f"{raw.at[i, 'location']} | {raw.at[i, 'seniority_level']}")
        if SAVE and n % 10 == 0:
            raw.to_csv(RAW_OUT, index=False)             # save progress every 10 rows
finally:
    driver.quit()
    if SAVE:
        raw.to_csv(RAW_OUT, index=False)
        print("Saved to", RAW_OUT)

Rows needing repair: 272
   DEBUG details -> seniority: '' | type: '' | industries: '' | description length: 373
[1/3] Data Scientist-Data Science-Data Scientist | EXL | Gurugram, Haryana, India | nan
   DEBUG details -> seniority: '' | type: '' | industries: '' | description length: 1866
[2/3] Senior Data Scientist: 978 | C5i | Hyderabad, Telangana, India | nan
   DEBUG details -> seniority: '' | type: '' | industries: '' | description length: 1523
[3/3] Senior Data Scientist | LatentView Analytics | Chennai, Tamil Nadu, India | nan


In [9]:
import shutil
from pathlib import Path

ROOT = Path(r"C:\Linkedin_Job_Analysis")

RAW = ROOT / "DATA" / "jobs_raw.csv"
MAX_ROWS = None          # None = repair everything. Use 5 for a quick trial.
LABELS = {"seniority level": "seniority_level",
          "employment type": "employment_type",
          "industries": "industries"}


def txt(parent, selector):
    try:
        el = parent.find_element(By.CSS_SELECTOR, selector)
        return re.sub(r"\s+", " ", el.get_attribute("textContent") or "").strip()
    except NoSuchElementException:
        return ""


def header_from_page_title(page_title):
    """'C5i hiring Senior Data Scientist in Hyderabad, India | LinkedIn' -> title, company, location"""
    t = page_title.replace(" | LinkedIn", "").strip()
    company = title = location = ""
    if " hiring " in t:
        company, t = t.split(" hiring ", 1)
    if " in " in t:
        title, location = t.rsplit(" in ", 1)
    else:
        title = t
    return {"title": title.strip(), "company": company.strip(), "location": location.strip()}


def read_criteria(driver):
    for attempt in range(3):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        found = {}
        for item in driver.find_elements(By.CSS_SELECTOR, "li.description__job-criteria-item"):
            label = txt(item, "h3.description__job-criteria-subheader").lower()
            if label in LABELS:
                found[LABELS[label]] = txt(item, "span.description__job-criteria-text")
        if found:
            return found
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(2)
    return {}


shutil.copy(RAW, ROOT / "DATA" / "jobs_raw_backup.csv")          # safety copy
raw = pd.read_csv(RAW, dtype={"job_id": str})
need_cols = ["title", "company", "location", "seniority_level", "employment_type", "industries"]
todo = list(raw.index[raw[need_cols].isna().any(axis=1)])
if MAX_ROWS:
    todo = todo[:MAX_ROWS]
print("Rows to repair:", len(todo))

driver = webdriver.Chrome()
driver.set_window_size(1400, 1000)
empty_streak = 0
try:
    for n, i in enumerate(todo, start=1):
        driver.get(raw.at[i, "job_url"])
        time.sleep(random.uniform(2, 3))
        u = driver.current_url.lower()
        if "authwall" in u or "/login" in u or "/checkpoint" in u:
            print("Login wall appeared. Stopping safely; progress is saved.")
            break

        info = header_from_page_title(driver.title)
        dom_title = txt(driver, "h1.top-card-layout__title") or txt(driver, "h2.top-card-layout__title")
        if dom_title:
            info["title"] = dom_title
        info.update(read_criteria(driver))

        got = 0
        for col, val in info.items():
            if val and pd.isna(raw.at[i, col]):
                raw.at[i, col] = val
                got += 1
        empty_streak = empty_streak + 1 if got == 0 else 0

        print(f"[{n}/{len(todo)}] filled {got} | {raw.at[i, 'title']} | {raw.at[i, 'company']} | {raw.at[i, 'seniority_level']}")
        if n % 10 == 0:
            raw.to_csv(RAW, index=False)
        if empty_streak >= 8:
            print("8 rows in a row gave nothing. Stopping so we do not waste time.")
            break
finally:
    driver.quit()
    raw.to_csv(RAW, index=False)
    print("\nSaved. Still missing per column:")
    print(raw[need_cols].isna().sum())

Rows to repair: 225
[1/225] filled 3 | Senior Data Scientist | Aditya Birla Capital | Mid-Senior level
[2/225] filled 3 | Senior Data Scientist | Capco | Mid-Senior level
[3/225] filled 3 | Data Scientist, Actimize | NiCE | Mid-Senior level
[4/225] filled 3 | Senior Data Scientist – Search | SproutsAI | Mid-Senior level
[5/225] filled 3 | Senior Data Scientist | IDFC FIRST Bank | Mid-Senior level
[6/225] filled 3 | Data Scientist Executive | GSTSahay | Mid-Senior level
[7/225] filled 3 | Data Scientist II | Bristol Myers Squibb | Entry level
[8/225] filled 3 | Data Scientist | Tata Consultancy Services | Not Applicable
[9/225] filled 3 | Senior Data Scientist | nference | Mid-Senior level
[10/225] filled 3 | Data Scientist | Thales | Not Applicable
[11/225] filled 3 | Data Scientist - Data Modeling/analytics | Amgen | Not Applicable
[12/225] filled 3 | Principal Data Scientist | Atlassian | Mid-Senior level
[13/225] filled 3 | Data Scientist - R01572015 | Brillio | Mid-Senior level
[14

In [11]:
RAW = ROOT / "DATA" / "jobs_raw.csv"
MAX_ROWS = None     # 3 = trial (nothing saved).  None = repair all 56 and save.


def txt(selector):
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        return re.sub(r"\s+", " ", el.get_attribute("textContent") or "").strip()
    except NoSuchElementException:
        return ""


raw = pd.read_csv(RAW, dtype={"job_id": str})
todo = list(raw.index[raw["company"].isna() | raw["location"].isna()])
print("Rows missing company/location:", len(todo))
if MAX_ROWS:
    todo = todo[:MAX_ROWS]
else:
    shutil.copy(RAW, ROOT / "DATA" / "jobs_raw_backup2.csv")

driver = webdriver.Chrome()
driver.set_window_size(1400, 1000)
try:
    for n, i in enumerate(todo, start=1):
        driver.get(raw.at[i, "job_url"])
        time.sleep(random.uniform(3, 4))
        u = driver.current_url.lower()
        if "authwall" in u or "/login" in u or "/checkpoint" in u:
            print("Login wall appeared. Stopping safely.")
            break

        company = txt("a.topcard__org-name-link") or txt(".topcard__flavor")
        location = txt("span.topcard__flavor--bullet")
        print(f"[{n}/{len(todo)}] page title: {driver.title}")
        print(f"      company: {company!r} | location: {location!r}")

        if company and pd.isna(raw.at[i, "company"]):
            raw.at[i, "company"] = company
        if location and pd.isna(raw.at[i, "location"]):
            raw.at[i, "location"] = location
        if MAX_ROWS is None and n % 10 == 0:
            raw.to_csv(RAW, index=False)
finally:
    driver.quit()
    if MAX_ROWS is None:
        raw.to_csv(RAW, index=False)
        print("\nSaved. Still missing:")
        print(raw[["company", "location"]].isna().sum())

Rows missing company/location: 56
[1/56] page title: Amazon hiring Data Engineer-I, SmartCommerce in Bengaluru, Karnataka, India | LinkedIn
      company: 'Amazon' | location: 'Bengaluru, Karnataka, India'
[2/56] page title: Amazon hiring Data Engineer I, CMT in Bengaluru, Karnataka, India | LinkedIn
      company: 'Amazon' | location: 'Bengaluru, Karnataka, India'
[3/56] page title: EXL hiring Azure Data Engineer in Pune Division, Maharashtra, India | LinkedIn
      company: 'EXL' | location: 'Pune Division, Maharashtra, India'
[4/56] page title: SourcingXPress hiring Senior Data Engineer in Ahmedabad, Gujarat, India | LinkedIn
      company: 'SourcingXPress' | location: 'Ahmedabad, Gujarat, India'
[5/56] page title: Infosys hiring AWS Data Engineer in Bengaluru East, Karnataka, India | LinkedIn
      company: 'Infosys' | location: 'Bengaluru East, Karnataka, India'
[6/56] page title: UST hiring Lead II - Data Engineering in Trivandrum, Kerala, India | LinkedIn
      company: 'UST' | 